**Experiment Orchestrator – Quick Guide**

---

### Purpose

To automate and parallelize the execution of the entire sensitivity analysis. This script reads the main `config.py` file, generates all necessary experiment jobs, and runs them using `papermill`.

---

### Workflow

1.  **Generate Job Queues (`create_job_queues`)**
    *   Reads `config.py` to create a list of all `(model, configuration)` experiments.
    *   Crucially, it separates jobs into two distinct lists: `regular_jobs` and `heavy_jobs` (e.g., `N-Beats`), based on the `HEAVY_MODELS` definition.

2.  **Execute Phase 1: Regular Jobs**
    *   Launches the `regular_jobs` in parallel, running up to the `MAX_CONCURRENT_JOBS_REGULAR` limit.
    *   The script waits until all of these jobs are complete before proceeding.

3.  **Execute Phase 2: Heavy Jobs**
    *   After Phase 1 is finished, it launches the `heavy_jobs`.
    *   These jobs are run with their own, lower concurrency limit (`MAX_CONCURRENT_JOBS_HEAVY`) to manage memory and resource usage.

4.  **Generate Outputs**
    *   For each job, it saves three artifacts in the `papermill_outputs/` directory:
        *   A **results CSV** in the `results/` folder.
        *   An **executed notebook** for visual inspection in the `notebooks/` folder.
        *   A **log file** for debugging in the `logs/` folder.

---

### Quick Configuration

All settings are at the top of the `run_all_experiments.py` script.

```python
# --- Concurrency Settings ---
MAX_CONCURRENT_JOBS_REGULAR = 12
MAX_CONCURRENT_JOBS_HEAVY = 2 

# --- Experiment Settings ---
DATASET_NAMES_TO_RUN = ['UNISIM', 'VOLVE', 'OPSD']

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import torch
torch.set_num_threads(1)

import subprocess
import time
from pathlib import Path
from typing import List, Dict, Any
from itertools import product
from config import HYPERPARAM_DESCRIPTIONS, CONFIG_DESCRIPTIONS

# ==============================================================================
# 1. User Configuration (EDIT THESE VALUES)
# ==============================================================================
TEMPLATE_NOTEBOOK = "experiment_template.ipynb"

# --- Concurrency Settings ---
# Max jobs for regular, lightweight models
MAX_CONCURRENT_JOBS_REGULAR = 12
# Max jobs for heavy, memory-intensive models
MAX_CONCURRENT_JOBS_HEAVY = 2 

# --- Experiment Settings ---
DATASET_NAMES_TO_RUN = ['UNISIM', 'VOLVE', 'OPSD']
# DATASET_NAMES_TO_RUN = ['VOLVE']
# --- Output Directories ---
OUTPUT_DIR = Path("papermill_outputs")
RESULTS_DIR = OUTPUT_DIR / "results"
LOGS_DIR = OUTPUT_DIR / "logs"
NOTEBOOKS_DIR = OUTPUT_DIR / "notebooks"

# ==============================================================================
# 2. Job Generation Logic (Unchanged from your version)
# ==============================================================================

def create_job_queues() -> tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Generates two queues of jobs:
    1. regular_jobs: Standard jobs that can run with high concurrency.
    2. heavy_jobs: Jobs that must run with limited concurrency.
    """
    regular_jobs = []
    heavy_jobs = []
    
    HEAVY_MODELS = ["N-Beats"]

    for model_type, config_size in product(HYPERPARAM_DESCRIPTIONS.keys(), CONFIG_DESCRIPTIONS.keys()):
        if config_size not in HYPERPARAM_DESCRIPTIONS.get(model_type, {}):
            continue
            
        job_id = f"{model_type}-{config_size.replace(' ', '_').replace('&', 'and')}"
        params = {
            "model_type": model_type, "model_config_size": config_size,
            "DATASET_NAMES_TO_RUN": DATASET_NAMES_TO_RUN,
            "output_csv_path": str(RESULTS_DIR / f"results-{job_id}.csv")
        }
        job_details = {
            "id": job_id, "params": params,
            "output_nb_path": NOTEBOOKS_DIR / f"output-{job_id}.ipynb",
            "log_path": LOGS_DIR / f"log-{job_id}.txt"
        }
        
        if model_type in HEAVY_MODELS:
            heavy_jobs.append(job_details)
        else:
            regular_jobs.append(job_details)
            
    return regular_jobs, heavy_jobs

# ==============================================================================
# 3. Main Orchestration Logic (Now a reusable function)
# ==============================================================================

def run_job_phase(job_queue: List[Dict], max_concurrent: int, phase_name: str):
    """
    A generic function to run a queue of jobs with a specific concurrency limit.
    """
    if not job_queue:
        print(f"Phase '{phase_name}': No jobs to run.")
        return

    print(f"🚀 Starting PHASE: {phase_name} ({len(job_queue)} jobs)")
    print(f"⚙️  Maximum of {max_concurrent} concurrent processes for this phase.")
    print("-" * 60)

    active_processes: Dict[subprocess.Popen, tuple] = {}
    
    # Make a copy to not modify the original list
    jobs_to_run = job_queue.copy()

    while jobs_to_run or active_processes:
        while len(active_processes) < max_concurrent and jobs_to_run:
            job = jobs_to_run.pop(0)
            cmd = ["papermill", TEMPLATE_NOTEBOOK, str(job["output_nb_path"]), "--log-output", "--progress-bar"]
            for key, val in job["params"].items():
                param_val = ",".join(val) if isinstance(val, list) else str(val)
                cmd.extend(["-p", key, param_val])
            
            print(f"✨ [{phase_name} LAUNCH] Launching job: {job['id']}")
            log_file = open(job["log_path"], "w")
            proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
            active_processes[proc] = (job, log_file)

        for proc, (job, log_file) in list(active_processes.items()):
            if proc.poll() is not None:
                return_code = proc.returncode
                status = "✅ COMPLETED" if return_code == 0 else "❌ FAILED"
                print(f"🏁 [FINISH] Job {job['id']} finished with status: {status} (Code: {return_code})")
                log_file.close()
                del active_processes[proc]
        
        time.sleep(5)

    print(f"\n--- Phase '{phase_name}' complete ---")

def main():
    """
    Main function to run the orchestration in two configurable phases.
    """
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    NOTEBOOKS_DIR.mkdir(parents=True, exist_ok=True)

    regular_jobs, heavy_jobs = create_job_queues()

    # --- Run each phase with its own concurrency limit ---
    run_job_phase(regular_jobs, MAX_CONCURRENT_JOBS_REGULAR, "Regular Models")
    run_job_phase(heavy_jobs, MAX_CONCURRENT_JOBS_HEAVY, "Heavy Models (N-Beats)")
        
    print("\n🎉 Orchestration complete.")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n🛑 Orchestration interrupted by user.")

🚀 Starting PHASE: Regular Models (5 jobs)
⚙️  Maximum of 12 concurrent processes for this phase.
------------------------------------------------------------
✨ [Regular Models LAUNCH] Launching job: AutoARIMA-Small_and_Fast
✨ [Regular Models LAUNCH] Launching job: AutoARIMA-Medium_(Balanced)
✨ [Regular Models LAUNCH] Launching job: AutoARIMA-Large_and_Robust
✨ [Regular Models LAUNCH] Launching job: AutoARIMA-Stable_and_Regularized
✨ [Regular Models LAUNCH] Launching job: AutoARIMA-Wide_and_Shallow
🏁 [FINISH] Job AutoARIMA-Medium_(Balanced) finished with status: ✅ COMPLETED (Code: 0)
🏁 [FINISH] Job AutoARIMA-Wide_and_Shallow finished with status: ✅ COMPLETED (Code: 0)
🏁 [FINISH] Job AutoARIMA-Small_and_Fast finished with status: ✅ COMPLETED (Code: 0)
🏁 [FINISH] Job AutoARIMA-Large_and_Robust finished with status: ✅ COMPLETED (Code: 0)
🏁 [FINISH] Job AutoARIMA-Stable_and_Regularized finished with status: ✅ COMPLETED (Code: 0)

--- Phase 'Regular Models' complete ---
Phase 'Heavy Models (